# Homework 03: Machine Learning Model Building and Tracking with MLflow

Author: Rongshan Wei 

UNI: rw3082

Date: April 13th, 2026

## Project Overview
This notebook focuses on the end-to-end Machine Learning (ML) lifecycle using the Formula 1 (F1) historical dataset. The primary objective is to build a predictive model—leveraging features such as driver performance, constructor data, and race conditions—while utilizing MLflow for systematic experiment tracking. By the end of this project, we will have executed at least 10 distinct experiments with varying hyperparameters to identify the optimal model for F1 outcome prediction.

## Technical Stack
* Platform: Databricks (Cloud-based Spark environment)
* ML Engine: Scikit-learn / XGBoost (Predictive modeling with tunable hyperparameters)
* Experiment Tracking: MLflow (Logging parameters, metrics, models, and artifacts)
* Version Control: GitHub (Frequent commits to track development progress)

## Dataset Description
The analysis leverages F1 datasets sourced from AWS S3, allowing for complex relational joins to create a feature-rich training set:
* drivers: Personal information and unique identifiers for all F1 drivers.
* results: Finishing positions, points, and status for every race.
* pit_stops: Granular data on every pit stop event, including duration.
* races: Contextual information about each Grand Prix (date, location, year).

## Methodology & Best Practices
In adherence to industrial coding standards and the specific requirements of this assignment, each iteration is managed via the following framework:
* Logic Formulation: Defining the ML problem (Regression/Classification) and feature selection strategy.
* Implementation & Tracking: * Hyperparameters: Logging specific model configurations (e.g., learning rate, depth).
  * Metrics: Tracking performance indicators like RMSE, MAE, or Accuracy.
  * Artifacts: Saving visual diagnostics (Residual plots, Feature Importance) and CSV results.
* Model Selection: Utilizing the MLflow UI to compare runs and justify the selection of the "Best Model" based on logged metrics.

## I. Global Configuration & Environment
Before processing large-scale datasets, it is a Best Practice to document the computational environment to ensure reproducibility.
* Spark Version: 3.x (Databricks Runtime)
* Cluster Configuration: Serverless Compute / Standard Runtime
* ML Engine: Scikit-learn (compatible with MLflow Autologging)
* Tracking Server: Databricks Managed MLflow
* Primary Language: Python 3.10+

## II. Library Ingestion

1. Logic Formulation

Following PEP 8 guidelines, all imports are centralized at the top of the notebook. In this iteration, I have expanded the library stack to include:

* MLflow: For tracking hyperparameters, metrics, and saving model artifacts.
* Scikit-Learn: To handle data splitting and model construction.
* Visualization (Matplotlib/Seaborn): To generate the required diagnostic plots (artifacts) as specified in the assignment requirements.
* Standard PySpark Modules: Aliased as F to maintain a clean namespace while processing large-scale F1 data.

2. Implementation

In [0]:
# --- Standard Library Imports ---
import os
import sys

# --- Third-party Library Imports: Data Processing ---
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Machine Learning & Visualization ---
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor  # Example choice for F1 modeling
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Set plot style for artifacts
plt.style.use('seaborn-v0_8-muted')

3. Code Walkthrough
* import mlflow: This is the core engine for this assignment. It allows us to wrap our training code in a start_run() block to capture every experiment detail.
* train_test_split: A critical step in Computational Thinking; we must isolate a portion of the data to validate the model's performance on unseen races.
* RandomForestRegressor: A robust choice for F1 data, as it handles the non-linear relationships between variables (like track temperature and pit stop duration) better than simple linear models.
* matplotlib & seaborn: These will be used to generate the Artifacts (such as Feature Importance plots or Residual charts) that must be logged to MLflow to satisfy the 20-point requirement.

## III. Data Ingestion & Feature Engineering

1. Logic Formulation
To build a high-quality predictive model, I will aggregate multiple relational tables from the F1 dataset to create a comprehensive feature set.
The Strategy:
* Core Table: Use results as the primary dataset containing our target variable (positionOrder).
* Contextual Joins: Join with races to capture temporal/circuit factors, drivers for biographical features, and constructors to account for team technical superiority.
* Feature Selection: I will focus on attributes known to impact race outcomes: driver age, starting grid position, and historical team performance.
* Target Definition: We will prepare the data for a Regression task—predicting the final positionOrder.

In [0]:
# Configuration: AWS S3 / Databricks Volume Path
# Adjust this path based on your specific S3 mount or volume
DATA_PATH = '/Volumes/gr5069/raw/f1_data/'

# Load relevant datasets
df_results = spark.read.csv(f"{DATA_PATH}results.csv", header=True, inferSchema=True)
df_races = spark.read.csv(f"{DATA_PATH}races.csv", header=True, inferSchema=True)
df_drivers = spark.read.csv(f"{DATA_PATH}drivers.csv", header=True, inferSchema=True)
df_constructors = spark.read.csv(f"{DATA_PATH}constructors.csv", header=True, inferSchema=True)
df_status = spark.read.csv(f"{DATA_PATH}status.csv", header=True, inferSchema=True)

# Feature Engineering: Joining and selecting relevant columns
# We focus on the modern era (e.g., post-2010) for more consistent data patterns
ml_data = (
    df_results.select("raceId", "driverId", "constructorId", "grid", "positionOrder", "statusId")
    .join(df_races.select("raceId", "year", "circuitId"), on="raceId")
    .join(df_drivers.select("driverId", "dob"), on="driverId")
    .join(df_constructors.select("constructorId", "name"), on="constructorId")
    # Filter for completed races to reduce noise from random mechanical failures
    .filter(F.col("statusId") == 1) 
)

# Convert to Pandas for Scikit-learn compatibility (standard practice for medium-sized F1 data)
final_df = ml_data.toPandas()

# Preliminary Data Profile
print(f"Total records for ML training: {len(final_df)}")
display(final_df.head(10))

3. Code Walkthrough
* inferSchema=True: As with previous assignments, ensuring grid and positionOrder are integers is critical for mathematical modeling.
* filter(F.col("statusId") == 1): This is a Computational Thinking decision. By focusing on drivers who actually finished the race (statusId=1), we remove the "luck" factor of engine blowouts or crashes, allowing the model to focus on pure performance prediction.
* toPandas(): Since we are using Scikit-Learn (as shown in the teacher's example), converting the Spark DataFrame to a local Pandas DataFrame is necessary for the training functions.